In [1]:
import requests
import pandas as pd
import time
import os
from datetime import datetime, timedelta

def fetch_mid_chunk(start_iso, end_iso):
    base_url = "https://data.elexon.co.uk/bmrs/api/v1/balancing/pricing/market-index"
    params = {'from': start_iso, 'to': end_iso, 'format': 'json'}
    headers = {'accept': 'application/json'}

    try:
        response = requests.get(base_url, params=params, headers=headers)
        if response.status_code == 200:
            data = response.json()
            return pd.DataFrame(data if isinstance(data, list) else data.get('data', []))
        return pd.DataFrame()
    except Exception:
        return pd.DataFrame()

def process_combined_price(df):
    """
    Groups data by settlement period and calculates the 
    Volume Weighted Average Price (VWAP) across providers.
    """
    if df.empty:
        return df

    # 1. Identify the key columns (assuming 'startTime', 'price', and 'volume')
    # Note: Column names in the API response are usually 'startTime', 'price', 'volume'
    # We group by startTime to align APX and N2EX for the same period.
    
    def vwap(group):
        total_volume = group['volume'].sum()
        if total_volume == 0:
            return pd.Series({'combined_price': group['price'].mean(), 'total_volume': 0})
        
        # Weighted Price calculation
        weighted_price = (group['price'] * group['volume']).sum() / total_volume
        return pd.Series({'combined_price': weighted_price, 'total_volume': total_volume})

    # 2. Apply the calculation
    processed = df.groupby('startTime').apply(vwap, include_groups=False).reset_index()
    return processed

def get_yearly_data(year):
    all_chunks = []
    current_start = datetime(year, 1, 1, 0, 0)
    year_end = datetime(year, 12, 31, 23, 59)

    while current_start < year_end:
        current_end = min(current_start + timedelta(days=7), year_end)
        start_str = current_start.strftime("%Y-%m-%dT%H:%M:%SZ")
        end_str = current_end.strftime("%Y-%m-%dT%H:%M:%SZ")
        
        print(f"  Fetching: {start_str} to {end_str}")
        df_chunk = fetch_mid_chunk(start_str, end_str)
        
        if not df_chunk.empty:
            all_chunks.append(df_chunk)
        
        current_start = current_end + timedelta(minutes=1)
        time.sleep(0.2)

    if all_chunks:
        full_year_df = pd.concat(all_chunks, ignore_index=True)
        # --- NEW STEP: WEIGHTED AVERAGE PROCESSING ---
        print(f"  Calculating weighted averages for {year}...")
        return process_combined_price(full_year_df)
    return None

def main():
    try:
        start_yr = int(input("Enter start year: "))
        end_yr = int(input("Enter end year: "))
    except ValueError:
        return

    # A year is only "complete" once it has fully elapsed. The current
    # (in-progress) year is always partial, so it should be refetched
    # even if a file for it already exists, to keep it up to date.
    current_year = datetime.now().year

    for year in range(start_yr, end_yr + 1):
        filename = f"elexon_MID_{year}.csv"
        is_complete_year = year < current_year

        if os.path.exists(filename) and is_complete_year:
            print(f"--- Skipping {year}: File exists and year is complete. ---")
            continue

        if os.path.exists(filename) and not is_complete_year:
            print(f"--- Refetching {year}: existing file is a partial year, updating with latest data. ---")

        yearly_df = get_yearly_data(year)

        if yearly_df is not None and not yearly_df.empty:
            yearly_df.to_csv(filename, index=False)
            print(f"--- Saved {year} (VWAP processed) ---\n")

if __name__ == "__main__":
    main()

--- Refetching 2026: existing file is a partial year, updating with latest data. ---
  Fetching: 2026-01-01T00:00:00Z to 2026-01-08T00:00:00Z
  Fetching: 2026-01-08T00:01:00Z to 2026-01-15T00:01:00Z
  Fetching: 2026-01-15T00:02:00Z to 2026-01-22T00:02:00Z
  Fetching: 2026-01-22T00:03:00Z to 2026-01-29T00:03:00Z
  Fetching: 2026-01-29T00:04:00Z to 2026-02-05T00:04:00Z
  Fetching: 2026-02-05T00:05:00Z to 2026-02-12T00:05:00Z
  Fetching: 2026-02-12T00:06:00Z to 2026-02-19T00:06:00Z
  Fetching: 2026-02-19T00:07:00Z to 2026-02-26T00:07:00Z
  Fetching: 2026-02-26T00:08:00Z to 2026-03-05T00:08:00Z
  Fetching: 2026-03-05T00:09:00Z to 2026-03-12T00:09:00Z
  Fetching: 2026-03-12T00:10:00Z to 2026-03-19T00:10:00Z
  Fetching: 2026-03-19T00:11:00Z to 2026-03-26T00:11:00Z
  Fetching: 2026-03-26T00:12:00Z to 2026-04-02T00:12:00Z
  Fetching: 2026-04-02T00:13:00Z to 2026-04-09T00:13:00Z
  Fetching: 2026-04-09T00:14:00Z to 2026-04-16T00:14:00Z
  Fetching: 2026-04-16T00:15:00Z to 2026-04-23T00:15:00Z
  F